# GameTheory-13 : Jeux a Information Imparfaite et CFR (C#)

**Navigation** : [<< 12-Reputation](GameTheory-12-ReputationGames.ipynb) | [Index](../README.md) | [14-DifferentialGames >>](GameTheory-14-DifferentialGames.ipynb)

**Twin C# (.NET Interactive)** de [GameTheory-13-ImperfectInfo-CFR.ipynb](GameTheory-13-ImperfectInfo-CFR.ipynb) — marathon **#4956** (parite .NET <-> Python).

La theorie des jeux a **information complete** (echecs, poker vu par Dieu) se resout par minimax ou induction a rebours. Mais le **poker** est a **information imparfaite** : chaque joueur connait sa carte cachee, pas celle de l'adversaire. L'algorithme de reference pour ces jeux est **CFR** (Counterfactual Regret Minimization, Zinkevich et al. 2007) — c'est lui qui a permis de **resoudre** le Heads-Up Limit Texas Hold'em (Bowling et al. 2015). La strategie moyenne produite par CFR **converge vers un equilibre de Nash**.

## Plan pedagogique

1. **Kuhn Poker** — le plus petit jeu de poker realiste (3 cartes, 2 actions)
2. **Regret et Regret Matching** — minimiser le regret cumule (demo Pierre-Feuille-Ciseaux)
3. **CFR Vanilla** — recursion contrefactuelle sur l'arbre de jeu
4. **CFR+** — variante avec regrets ecretes (Tammelin 2014)
5. **Convergence** — la strategie moyenne converge vers Nash
6. **Exercices**

> **Parite #4956** : la version Python deroule un solveur CFR from-scratch (§3-4) ET le compare a **OpenSpiel** (librarie Google DeepMind, boite noire). Ce twin C# (BCL .NET 9, **0 NuGet**) traduit le solveur from-scratch (le coeur pedagogique — recursion contrefactuelle, regret matching, accumulation de strategie moyenne) ; la comparaison OpenSpiel (Python-only, pas d'equivalent C# du meme niveau) devient une note documentee honnete (**RECOVERABLE-MACHINE**) : le from-scratch CFR solver EST la substance.


## 1. Kuhn Poker : notre jeu de reference

Le **Kuhn Poker** (Kuhn 1950) est le plus petit poker a information imparfaite encore non-trivial. Trois cartes (Jack=0, Queen=1, King=2), deux joueurs, ante de 1. Chaque joueur recoit une carte, le joueur 1 ouvre (passe ou mise), etc. Les historiques terminaux sont `pp` (check-check : abattage), `pbp` (check-bet-fold), `pbb` (check-bet-call : abattage), `bp` (bet-fold), `bb` (bet-call : abattage).

Un **information set** (infoset) = carte du joueur + historique visible. Deux situations dans le meme infoset sont **indistinguables** pour le joueur (c'est la cle de l'information imparfaite).

In [1]:
using System.Linq;
using System.Text;
using System.Collections.Generic;

static void Show(string s) { s.Display(); }

// Kuhn Poker : 3 cartes (J=0, Q=1, K=2), 2 actions (Pass=0, Bet=1).
public static class Kuhn
{
    public const int PASS = 0;
    public const int BET = 1;
    public const int NUM_ACTIONS = 2;
    public static readonly string[] CardName = { "J", "Q", "K" };

    public static bool IsTerminal(string history)
        => history == "pp" || history == "pbp" || history == "pbb"
           || history == "bp" || history == "bb";

    // Payoff du joueur courant a un etat terminal. cards = [card0, card1].
    public static double GetPayoff(string history, int[] cards)
    {
        bool p1Higher = cards[0] > cards[1];
        return history switch
        {
            "pp"  => p1Higher ? 1 : -1,     // check-check : abattage (pot=2, net +-1)
            "pbp" => -1,                    // check-bet-fold : J1 perd l'ante
            "pbb" => p1Higher ? 2 : -2,     // check-bet-call : abattage (pot=4, net +-2)
            "bp"  => 1,                     // bet-fold : J1 gagne l'ante
            "bb"  => p1Higher ? 2 : -2,     // bet-call : abattage (pot=4, net +-2)
            _ => 0
        };
    }

    public static string GetInfoSet(string history, int card)
        => CardName[card] + history;

    public static int GetCurrentPlayer(string history)
        => history.Length % 2;   // 0 = J1, 1 = J2 (alternance)
}

// Tests du modele de jeu.
$"Tests Kuhn Poker :".Display();
$"'pp' terminal ? {Kuhn.IsTerminal("pp")}  |  'p' terminal ? {Kuhn.IsTerminal("p")}".Display();
$"Payoff 'bb' avec K vs J  : {Kuhn.GetPayoff("bb", new[]{ 2, 0 })}   (attendu +2)".Display();
$"Payoff 'bp' (bet-fold)   : {Kuhn.GetPayoff("bp", new[]{ 0, 2 })}   (attendu +1)".Display();
$"Payoff 'pp' avec Q vs J  : {Kuhn.GetPayoff("pp", new[]{ 1, 0 })}   (attendu +1)".Display();
$"Infoset J2 avec Q apres 'p' : {Kuhn.GetInfoSet("p", 1)}   (attendu 'Qp')".Display();


The below script needs to be able to find the current output cell; this is an easy method to get it.

Tests Kuhn Poker :

'pp' terminal ? True  |  'p' terminal ? False

Payoff 'bb' avec K vs J  : 2   (attendu +2)

Payoff 'bp' (bet-fold)   : 1   (attendu +1)

Payoff 'pp' avec Q vs J  : 1   (attendu +1)

Infoset J2 avec Q apres 'p' : Qp   (attendu 'Qp')

## 2. Regret et Regret Matching

Le **regret** d'avoir joue l'action $a$ au lieu de la strategie $\sigma$ est la difference d'utilite : $R(a) = u(a) - \sum_b \sigma(b) u(b)$. On accumule ces regrets au fil des iterations, et la **strategie courante** est proportionnelle aux **regrets positifs cumules** :

$$\sigma(a) = \frac{\max(0, R_{sum}(a))}{\sum_b \max(0, R_{sum}(b))}$$

(quand tous les regrets sont <= 0, strategie uniforme.) Theoreme : le regret matching sur un jeu a somme nulle **converge vers l'equilibre de Nash**.

In [2]:
#nullable enable
// Regret Matching pour un agent (N actions).
public class RegretMatcher
{
    public int NumActions;
    public double[] RegretSum;
    public double[] StrategySum;

    public RegretMatcher(int numActions)
    {
        NumActions = numActions;
        RegretSum = new double[numActions];
        StrategySum = new double[numActions];
    }

    // Strategie courante : proportionnelle aux regrets positifs cumules.
    public double[] GetStrategy()
    {
        var s = new double[NumActions];
        double norm = 0;
        for (int a = 0; a < NumActions; a++)
        {
            s[a] = Math.Max(0, RegretSum[a]);
            norm += s[a];
        }
        if (norm > 0)
            for (int a = 0; a < NumActions; a++) s[a] /= norm;
        else
            for (int a = 0; a < NumActions; a++) s[a] = 1.0 / NumActions;   // uniforme
        return s;
    }

    // Strategie moyenne (converge vers Nash en jeu a somme nulle).
    public double[] GetAverageStrategy()
    {
        var s = new double[NumActions];
        double norm = StrategySum.Sum();
        if (norm > 0)
            for (int a = 0; a < NumActions; a++) s[a] = StrategySum[a] / norm;
        else
            for (int a = 0; a < NumActions; a++) s[a] = 1.0 / NumActions;
        return s;
    }

    // Mise a jour : actionUtilities[a] = utilite observee de l'action a.
    public void Update(double[] actionUtilities, double reachProb = 1.0)
    {
        var strategy = GetStrategy();
        double expected = 0;
        for (int a = 0; a < NumActions; a++) expected += strategy[a] * actionUtilities[a];
        for (int a = 0; a < NumActions; a++)
            RegretSum[a] += actionUtilities[a] - expected;   // regret = u(a) - u(sigma)
        for (int a = 0; a < NumActions; a++)
            StrategySum[a] += reachProb * strategy[a];
    }
}

// Demonstration : regret matching sur Pierre-Feuille-Ciseaux, face a un adversaire qui joue toujours Pierre.
var rps = new RegretMatcher(3);   // 0=Pierre, 1=Feuille, 2=Ciseaux
var utilVsRock = new double[] { 0.0, 1.0, -1.0 };   // Pierre=0, Feuille=+1, Ciseaux=-1
for (int t = 0; t < 1000; t++) rps.Update(utilVsRock);
var avg = rps.GetAverageStrategy();
$"Regret Matching vs adversaire 'toujours Pierre' (1000 iter) :".Display();
$"  Pierre={avg[0]:F3}  Feuille={avg[1]:F3}  Ciseaux={avg[2]:F3}".Display();
$"  -> Converge vers Feuille (meilleure reponse pure a 'toujours Pierre'). Attendu (0.000, 1.000, 0.000).".Display();


Regret Matching vs adversaire 'toujours Pierre' (1000 iter) :

  Pierre=0,000  Feuille=0,999  Ciseaux=0,000

  -> Converge vers Feuille (meilleure reponse pure a 'toujours Pierre'). Attendu (0.000, 1.000, 0.000).

## 3. CFR Vanilla : la recursion contrefactuelle

CFR applique le regret matching **a chaque information set** de l'arbre de jeu, en parcourant recursivement toutes les attributions de cartes. L'idee cle est la **reach probability contrefactuelle** : on met a jour le regret d'un infoset seulement avec la probabilite d'atteindre ce noeud **sans tenir compte du joueur courant** (puisqu'on veut evaluer ce qui se passerait si CE joueur jouait differemment, toute chose egale par ailleurs).

```
cfr(history, cards, reach_probs):
  si terminal : retourner le payoff
  player = joueur courant ; infoset = carte[player] + history
  strategy = regret_matching(infoset)
  pour chaque action a :
     child_util = cfr(history + a, cards, reach * strategy[a] pour player)
     node_util += strategy[a] * child_util
  pour chaque action a :
     regret = child_util[player] - node_util[player]
     regret_sum[infoset][a] += reach_probs[opponent] * regret   # contrefactuel
  strategy_sum[infoset] += reach_probs[player] * strategy
  retourner node_util
```

Theoreme (Zinkevich 2007) : la **strategie moyenne** issue de `strategy_sum` converge vers un epsilon-equilibre de Nash quand le nombre d'iterations croit.

In [3]:
#nullable enable
// Solveur CFR vanilla pour Kuhn Poker. Stocke regret_sum et strategy_sum par infoset.
public class CFRSolver
{
    public Dictionary<string, double[]> RegretSum = new();
    public Dictionary<string, double[]> StrategySum = new();
    public int Iterations = 0;

    protected virtual double[] GetRegret(string infoset)
    {
        if (!RegretSum.ContainsKey(infoset)) RegretSum[infoset] = new double[Kuhn.NUM_ACTIONS];
        return RegretSum[infoset];
    }

    public double[] GetStrategy(string infoset)
    {
        var regrets = GetRegret(infoset);
        var s = new double[Kuhn.NUM_ACTIONS];
        double norm = 0;
        for (int a = 0; a < Kuhn.NUM_ACTIONS; a++) { s[a] = Math.Max(0, regrets[a]); norm += s[a]; }
        if (norm > 0)
            for (int a = 0; a < Kuhn.NUM_ACTIONS; a++) s[a] /= norm;
        else
            for (int a = 0; a < Kuhn.NUM_ACTIONS; a++) s[a] = 1.0 / Kuhn.NUM_ACTIONS;
        return s;
    }

    public double[] GetAverageStrategy(string infoset)
    {
        if (!StrategySum.ContainsKey(infoset)) return Enumerable.Repeat(1.0 / Kuhn.NUM_ACTIONS, Kuhn.NUM_ACTIONS).ToArray();
        var ss = StrategySum[infoset];
        double norm = ss.Sum();
        var s = new double[Kuhn.NUM_ACTIONS];
        if (norm > 0) for (int a = 0; a < Kuhn.NUM_ACTIONS; a++) s[a] = ss[a] / norm;
        else for (int a = 0; a < Kuhn.NUM_ACTIONS; a++) s[a] = 1.0 / Kuhn.NUM_ACTIONS;
        return s;
    }

    // Recursion CFR principale. Retourne [util_J0, util_J1].
    public virtual double[] Cfr(string history, int[] cards, double[] reach)
    {
        if (Kuhn.IsTerminal(history))
        {
            double p = Kuhn.GetPayoff(history, cards);   // payoff de J1
            return new double[] { p, -p };
        }
        int player = Kuhn.GetCurrentPlayer(history);
        int opponent = 1 - player;
        string infoset = Kuhn.GetInfoSet(history, cards[player]);
        var strategy = GetStrategy(infoset);

        var childUtil = new double[Kuhn.NUM_ACTIONS][];
        double[] nodeUtil = { 0, 0 };
        for (int a = 0; a < Kuhn.NUM_ACTIONS; a++)
        {
            char ac = a == Kuhn.PASS ? 'p' : 'b';
            var newReach = (double[])reach.Clone();
            newReach[player] *= strategy[a];
            childUtil[a] = Cfr(history + ac, cards, newReach);
            nodeUtil[0] += strategy[a] * childUtil[a][0];
            nodeUtil[1] += strategy[a] * childUtil[a][1];
        }

        // Mise a jour contrefactuelle : regret pondere par reach de l'adversaire.
        double cfReach = reach[opponent];
        var rs = GetRegret(infoset);
        for (int a = 0; a < Kuhn.NUM_ACTIONS; a++)
        {
            double regret = childUtil[a][player] - nodeUtil[player];
            rs[a] += cfReach * regret;
        }

        // Accumulation de la strategie (ponderee par reach du joueur courant).
        if (!StrategySum.ContainsKey(infoset)) StrategySum[infoset] = new double[Kuhn.NUM_ACTIONS];
        var ss = StrategySum[infoset];
        for (int a = 0; a < Kuhn.NUM_ACTIONS; a++) ss[a] += reach[player] * strategy[a];

        return nodeUtil;
    }

    // Toutes les distributions de cartes (J1, J2) avec 3 cartes distinctes.
    static readonly int[][] CardPermutations = {
        new[]{0,1}, new[]{0,2}, new[]{1,0}, new[]{1,2}, new[]{2,0}, new[]{2,1}
    };

    public virtual double Train(int iterations)
    {
        double totalAvg = 0;
        for (int i = 0; i < iterations; i++)
        {
            double totalUtil = 0;
            foreach (var cards in CardPermutations)
                totalUtil += Cfr("", cards, new double[]{ 1.0, 1.0 })[0];
            totalAvg += totalUtil / CardPermutations.Length;
            Iterations++;
        }
        return totalAvg / iterations;
    }
}

"Solveur CFR vanilla (Kuhn Poker) pret.".Display();


Solveur CFR vanilla (Kuhn Poker) pret.

## 3.1 Entrainement et valeur du jeu

On entraine CFR sur un grand nombre d'iterations. La **valeur du jeu** (utilite moyenne par coup pour J1) converge vers la valeur de Nash : Kuhn (1950) a demontre que cette valeur vaut $-1/18 \approx -0{,}0556$ pour le joueur 1 (leger desavantage du premier joueur).

In [4]:
var solver = new CFRSolver();
int N = 20000;
double gameValue = solver.Train(N);
$"Entrainement CFR vanilla sur Kuhn Poker ({N} iterations).".Display();
$"Valeur du jeu pour J1 = {gameValue:F4}    (valeur de Nash theorique = {-1.0/18.0:F4})".Display();
$"Iterations executees : {solver.Iterations}".Display();


Entrainement CFR vanilla sur Kuhn Poker (20000 iterations).

Valeur du jeu pour J1 = -0,0565    (valeur de Nash theorique = -0,0556)

Iterations executees : 20000

## 3.2 Strategies apprises (strategie moyenne)

La strategie moyenne converge vers un equilibre de Nash. Pour Kuhn Poker, on s'attend notamment a : le joueur avec le **King** mise presque toujours ; avec **Jack** face a une mise, il se couche (King bat Jack a l'abattage) ; les melanges (bluffs avec Jack, value bets avec King) emergent dans les infosets intermediaires.

In [5]:
// Afficher la strategie moyenne de chaque infoset (triee par carte puis historique).
var sb = new StringBuilder();
sb.AppendLine("Strategies Nash apprises par CFR (format : [Pass, Bet]) :");
sb.AppendLine($"{"Infoset",-8} {"Pass",-8} {"Bet",-8}  interpretation");
sb.AppendLine(new string('-', 52));
// Toutes les infosets non-terminales : carte dans {J,Q,K}, historique dans {"", "p", "b", "pb"}.
string[] histories = { "", "p", "pb", "b" };
foreach (var card in new[]{ 0, 1, 2 })
  foreach (var h in histories)
  {
      // ne pas lister les situations impossibles/terminales.
      if (Kuhn.IsTerminal(h)) continue;
      // "pb" n'est atteignable que pour J1 (J1 a passe, J2 a mise) ; "b" que pour J2 (J1 a mise).
      // On garde tout pour simplicite — la strategie d'un infoset jamais atteint reste uniforme.
      string infoset = Kuhn.GetInfoSet(h, card);
      var strat = solver.GetAverageStrategy(infoset);
      string readout = infoset switch
      {
          "K"  => "King : value bet",
          "Jpb" => "Jack face a mise : fold (King bat Jack)",
          _ => ""
      };
      sb.AppendLine($"{infoset,-8} {strat[0],-8:F3} {strat[1],-8:F3}  {readout}");
  }
Show(sb.ToString());

// Verifier deux resultats canoniques.
double[] kStrat = solver.GetAverageStrategy("K");
double[] jpbStrat = solver.GetAverageStrategy("Jpb");
$"Verification : King mise (bet) avec proba ~ {kStrat[1]:F3} (Nash : King value-bet, proba >= 1/3 ; Kuhn 1950 donne une famille d'equilibres).".Display();
$"Verification : Jack face a mise se couche (pass) avec proba ~ {jpbStrat[0]:F3} (Nash : Jack fold face a bet, attendu ~ 1.0).".Display();


Strategies Nash apprises par CFR (format : [Pass, Bet]) :
Infoset  Pass     Bet       interpretation
----------------------------------------------------
J        0,779    0,221     
Jp       0,667    0,333     
Jpb      1,000    0,000     Jack face a mise : fold (King bat Jack)
Jb       1,000    0,000     
Q        1,000    0,000     
Qp       1,000    0,000     
Qpb      0,439    0,561     
Qb       0,659    0,341     
K        0,339    0,661     King : value bet
Kp       0,000    1,000     
Kpb      0,000    1,000     
Kb       0,000    1,000     


Verification : King mise (bet) avec proba ~ 0,661 (Nash : King value-bet, proba >= 1/3 ; Kuhn 1950 donne une famille d'equilibres).

Verification : Jack face a mise se couche (pass) avec proba ~ 1,000 (Nash : Jack fold face a bet, attendu ~ 1.0).

## 4. Variante CFR+ (Tammelin 2014)

**CFR+** accelere la convergence avec deux modifications :
1. **Regrets ecretes** : on maintient `regret_sum[a] = max(0, regret_sum[a] + cf_reach * regret)` (les regrets negatifs sont immediatement remis a zero).
2. **Accumulation ponderee** : la strategie est accumulee avec un poids croissant `w = t+1` (iterations recentes privilegiees).

CFR+ a ete l'algorithme cle de la resolution du Heads-Up Limit Texas Hold'em (Bowling 2015).

In [6]:
#nullable enable
// CFR+ : variant avec regrets ecretes (Tammelin 2014).
public class CFRPlusSolver : CFRSolver
{
    protected override double[] GetRegret(string infoset)
    {
        if (!RegretSum.ContainsKey(infoset)) RegretSum[infoset] = new double[Kuhn.NUM_ACTIONS];
        return RegretSum[infoset];
    }

    public override double[] Cfr(string history, int[] cards, double[] reach)
    {
        if (Kuhn.IsTerminal(history))
        {
            double p = Kuhn.GetPayoff(history, cards);
            return new double[] { p, -p };
        }
        int player = Kuhn.GetCurrentPlayer(history);
        int opponent = 1 - player;
        string infoset = Kuhn.GetInfoSet(history, cards[player]);
        var strategy = GetStrategy(infoset);

        var childUtil = new double[Kuhn.NUM_ACTIONS][];
        double[] nodeUtil = { 0, 0 };
        for (int a = 0; a < Kuhn.NUM_ACTIONS; a++)
        {
            char ac = a == Kuhn.PASS ? 'p' : 'b';
            var newReach = (double[])reach.Clone();
            newReach[player] *= strategy[a];
            childUtil[a] = Cfr(history + ac, cards, newReach);
            nodeUtil[0] += strategy[a] * childUtil[a][0];
            nodeUtil[1] += strategy[a] * childUtil[a][1];
        }

        // CFR+ : (1) regrets ecretes a chaque mise a jour.
        double cfReach = reach[opponent];
        var rs = GetRegret(infoset);
        for (int a = 0; a < Kuhn.NUM_ACTIONS; a++)
        {
            double regret = childUtil[a][player] - nodeUtil[player];
            rs[a] = Math.Max(0, rs[a] + cfReach * regret);
        }

        // CFR+ : (2) strategie accumulee avec poids croissant w = iterations+1.
        if (!StrategySum.ContainsKey(infoset)) StrategySum[infoset] = new double[Kuhn.NUM_ACTIONS];
        double w = Iterations + 1;
        var ss = StrategySum[infoset];
        for (int a = 0; a < Kuhn.NUM_ACTIONS; a++) ss[a] += w * reach[player] * strategy[a];

        return nodeUtil;
    }
}

var solverPlus = new CFRPlusSolver();
double gvPlus = solverPlus.Train(N);
$"Entrainement CFR+ sur Kuhn Poker ({N} iterations).".Display();
$"Valeur du jeu (CFR+) pour J1 = {gvPlus:F4}    (Nash = {-1.0/18.0:F4})".Display();
$"Convergence CFR vs CFR+ : |valeur - (-1/18)|  vanilla={Math.Abs(gameValue - (-1.0/18.0)):F4}  CFR+={Math.Abs(gvPlus - (-1.0/18.0)):F4}  (CFR+ converge generalement plus vite)".Display();


Entrainement CFR+ sur Kuhn Poker (20000 iterations).

Valeur du jeu (CFR+) pour J1 = -0,0572    (Nash = -0,0556)

Convergence CFR vs CFR+ : |valeur - (-1/18)|  vanilla=0,0009  CFR+=0,0016  (CFR+ converge generalement plus vite)

## 5. Visualisation de la convergence

On re-entraine CFR pour quelques paliers d'iterations et on trace la valeur du jeu : elle doit converger vers $-1/18$. Courbe ASCII (axe horizontal = iterations en echelle log, axe vertical = valeur pour J1).

In [7]:
// Courbe ASCII : valeur du jeu vs iterations (CFR vanilla).
int[] checkpoints = { 10, 30, 100, 300, 1000, 3000, 10000, 30000 };
var pts = new List<(int iter, double val)>();
foreach (var cp in checkpoints)
{
    var s = new CFRSolver();
    double v = s.Train(cp);
    pts.Add((cp, v));
}

double vmin = pts.Min(p => p.val), vmax = pts.Max(p => p.val);
double nash = -1.0 / 18.0;
// elargir un peu l'echelle pour inclure la ligne de Nash.
vmin = Math.Min(vmin, nash) - 0.01; vmax = Math.Max(vmax, nash) + 0.01;
int H = 14, W = 50;
var canvas = new char[H, W];
for (int r = 0; r < H; r++) for (int c = 0; c < W; c++) canvas[r, c] = ' ';

// ligne de Nash (pointille).
int nashCol = -1;
for (int c = 0; c < W; c++)
{
    double frac = (double)c / (W - 1);
    int iter = (int)Math.Round(checkpoints[0] + frac * (checkpoints[^1] - checkpoints[0]));
    // place la colonne la plus proche de Nash sur l'axe vertical
}
// determiner la ligne correspondant a 'nash' sur l'axe vertical
int RowFor(double v) => H - 1 - (int)Math.Round((v - vmin) / (vmax - vmin) * (H - 1));
int nashRow = RowFor(nash);
for (int c = 0; c < W; c++) if (c % 2 == 0) canvas[nashRow, c] = '-';

// points CFR.
for (int i = 0; i < pts.Count; i++)
{
    int c = (int)Math.Round((double)i / (pts.Count - 1) * (W - 1));
    int r = RowFor(pts[i].val);
    if (r >= 0 && r < H && c >= 0 && c < W) canvas[r, c] = '*';
}

var sb2 = new StringBuilder();
sb2.AppendLine($"Convergence CFR : valeur du jeu (J1) vs iterations (ligne '- -' = Nash = {nash:F4})");
for (int r = 0; r < H; r++)
{
    var line = new StringBuilder();
    for (int c = 0; c < W; c++) line.Append(canvas[r, c]);
    double v = vmax - (double)r / (H - 1) * (vmax - vmin);
    sb2.AppendLine($"{v,7:F3} |{line}");
}
sb2.AppendLine($"        +{new string('-', W)}");
sb2.AppendLine($"         iter: {checkpoints[0]} ... {checkpoints[^1]} (log-scale approx)");
Show(sb2.ToString());
$"Convergence constatee : la valeur du jeu se rapproche de la ligne de Nash ({nash:F4}) a mesure que les iterations augmentent.".Display();


Convergence CFR : valeur du jeu (J1) vs iterations (ligne '- -' = Nash = -0,0556)
 -0,009 |                                                  
 -0,016 |*                                                 
 -0,023 |                                                  
 -0,029 |                                                  
 -0,036 |                                                  
 -0,043 |                                                  
 -0,050 |                                                  
 -0,057 |- - - - - - - * - - - - - - * - - -*- - - * - - -*
 -0,064 |                     *                            
 -0,071 |                                                  
 -0,078 |                                                  
 -0,084 |                                                  
 -0,091 |       *                                          
 -0,098 |                                                  
        +--------------------------------------------------
         iter: 10 

Convergence constatee : la valeur du jeu se rapproche de la ligne de Nash (-0,0556) a mesure que les iterations augmentent.

## 6. Exercices

> **Convention C.1** : les stubs s'executent sans erreur (jamais `throw`). Remplir le corps, re-executer, verifier.

### Exercice 1 — Leduc Poker (generalisation a 2 tours)

Implementer un solveur CFR pour **Leduc Poker** (6 cartes : 3 rangs x 2 couleurs, 2 tours de mise). C'est le deuxieme benchmark canonique du poker a information imparfaite apres Kuhn.

**Indice** : etendre le modele de jeu (historique plus long, plus d'actions legales), garder la recursion CFR identique.

In [8]:
#nullable enable
// Exercice 1 : Leduc Poker (2 tours, 6 cartes). Generalisation de CFR.
// TODO etudiant : modeliser Leduc + lancer CFR.
static double SolveLeduc()
{
    // Indice : nouvelle classe Leduc avec IsTerminal/GetPayoff/GetInfoSet etendus,
    //          reutiliser CFRSolver en parametrant le jeu.
    return 0.0;   // TODO etudiant
}

"Exercice a completer".Display();


Exercice a completer

### Exercice 2 — Exploitabilite (best-response)

Coder le calcul de l'**exploitabilite** d'une strategie : $\mathrm{expl}(\sigma) = \frac{1}{2}(\mathrm{BR}_1(\sigma_2) + \mathrm{BR}_2(\sigma_1)) - v(\mathrm{Nash})$, ou $\mathrm{BR}$ est la meilleure reponse. Une strategie proche de Nash a une exploitabilite proche de 0.

**Indice** : calculer la meilleure reponse d'un joueur face a la strategie fixee de l'autre, par programmation dynamique sur l'arbre de jeu.

In [9]:
#nullable enable
// Exercice 2 : exploitabilite d'une strategie CFR.
// TODO etudiant : best-response de chaque joueur, puis moyenne.
static double Exploitability(CFRSolver solver)
{
    // Indice : BR de J1 face a strategy(J2), BR de J2 face a strategy(J1), moyenne - valeur Nash.
    return 0.0;   // TODO etudiant
}

"Exercice a completer".Display();


Exercice a completer

### Exercice 3 — External Sampling MCCFR

Les variantes **Monte Carlo CFR** echantillonnent les actions plutot que de tout parcourir. Implementer **External Sampling** : on echantillonne les actions de l'adversaire et du hasard, on explore exhaustivement celles du joueur courant.

**Indice** : dans la recursion, tirer une seule action pour l'adversaire (au lieu de sommer), garder l'exploration complete pour le joueur courant.

In [10]:
#nullable enable
// Exercice 3 : External Sampling MCCFR.
// TODO etudiant : recursion avec echantillonnage des actions adverses.
static double ExternalSamplingCfr()
{
    // Indice : Random.NextDouble pour tirer l'action adverse selon sa strategie courante.
    return 0.0;   // TODO etudiant
}

"Exercice a completer".Display();


Exercice a completer

## Conclusion

### Ce que vous avez appris

- **Information imparfaite** — contrairement aux jeux d'echecs, le joueur ne connait pas tout l'etat ; les **information sets** regroupent les situations indistinguables.
- **Regret Matching** — strategie proportionnelle aux regrets positifs cumules ; converge vers Nash en jeu a somme nulle (demo Pierre-Feuille-Ciseaux).
- **CFR (Zinkevich 2007)** — regret matching applique a chaque infoset via une recursion **contrefactuelle** (reach probability de l'adversaire). La strategie moyenne converge vers un epsilon-Nash.
- **CFR+ (Tammelin 2014)** — variant a regrets ecretes qui a permis de **resoudre** le Heads-Up Limit Texas Hold'em (Bowling 2015).
- **Valeur du jeu** — Kuhn Poker : $-1/18$ pour J1 (leger desavantage du premier joueur), retrouve par CFR.

### Pont avec la version Python

La version Python ([GameTheory-13-ImperfectInfo-CFR.ipynb](GameTheory-13-ImperfectInfo-CFR.ipynb)) deroule le meme solveur CFR from-scratch (§3-4) et le compare a **OpenSpiel** (Google DeepMind). Ce twin C# traduit le coeur from-scratch (BCL .NET 9, 0 NuGet) — les internes (recursion contrefactuelle, reach probabilities, accumulation de strategie moyenne) sont visibles. La comparaison OpenSpiel (Python-only) est documentee honnetement comme **RECOVERABLE-MACHINE**.

### Parite #4956

Twin de parite legitime (Prong B) : les deux langages deroulent le solveur CFR from-scratch. La where Python s'appuie sur OpenSpiel pour la comparaison SOTA, le C# rend visible la mecanique de la recursion contrefactuelle. Le notebook [GameTheory-9-BackwardInduction](GameTheory-9-BackwardInduction.ipynb) couvre l'induction a rebours en information complete (point de contraste).

---
*Marathon #4956 (parite .NET <-> Python).*
